# CryptoHypothesisLab — 05 / V2: sekundni Pre-Move Early Warning

**Potpuno nova Colab sesija, bez Google Drive mounta.** Analiza samo odobrenog privatnog Development ZIP paketa sa 1m OHLCV i rijetkim 5/15/60s C2 značajkama.

Pogotci, pogrešan smjer, bez pokreta, neprocjenjivi alarmi, propuštene *ex-post* epizode i grafikoni prije/tokom/nakon. **Nema C3 knjige, pravog OFI-ja, fillova, naknada, PnL-a ili potvrđenog edgea.**


In [ ]:
# 1. Sigurnosni preflight i javni kod, prije privatnog uploada.
import os,subprocess,sys,hashlib,json
from pathlib import Path
if any(Path(p).exists() for p in ('/content/drive/MyDrive','/content/gdrive/MyDrive','/gdrive/MyDrive')):
    raise RuntimeError('Drive je montiran. Zatvori runtime i kreiraj novu sesiju.')
if any(os.environ.get(k) for k in ('GOOGLE_APPLICATION_CREDENTIALS','GOOGLE_OAUTH_ACCESS_TOKEN','GOOGLE_DRIVE_TOKEN')):
    raise RuntimeError('Google credential detected; fail closed')
repo=Path('/content/hypolab-micro-research')
if repo.exists():raise RuntimeError('Koristi cistu sesiju')
subprocess.run(['git','clone','--depth','1','https://github.com/DuStER1921/CryptoHypothesisLab.git',str(repo)],check=True,stdout=subprocess.DEVNULL)
sys.path.insert(0,str(repo/'src'))
from hypolab.micro_bundle import inspect_micro_bundle,MAX_ARCHIVE_BYTES
from hypolab.micro_warning import make_micro_warning_plan,run_micro_warning_study
print('Analyst preflight PASS; Drive se NE montira.')


In [ ]:
# 2. Ucitaj samo svoj privatni MICRO ZIP iz racunara.
from google.colab import files
uploaded=files.upload()
if len(uploaded)!=1:raise RuntimeError('Tacno jedan MICRO ZIP je dozvoljen')
name,raw=next(iter(uploaded.items()))
if not name.startswith('HYPOLAB_MICRO_DEV_') or not name.endswith('.zip') or not 0<len(raw)<=MAX_ARCHIVE_BYTES:
    raise RuntimeError('Neispravna vrsta ili velicina ZIP paketa')
expected=input('Upisi SHA-256 MICRO ZIP-a iz pripremnog Colaba: ').strip().lower()
actual=hashlib.sha256(raw).hexdigest()
if actual!=expected:raise RuntimeError('SHA-256 mismatch; paket nije prihvacen')
private=Path('/content/hypolab-micro-input');private.mkdir(mode=0o700,exist_ok=False)
packet=private/'micro.zip'
with packet.open('xb') as f:
    os.chmod(packet,0o600)
    f.write(raw)
del uploaded,raw
manifest,bars,_=inspect_micro_bundle(packet,collect_5s=False)
print('MICRO ZIP VERIFICATION PASS:',manifest['month'],manifest['counts'])
print('C3 bookDepth: UNKNOWN_NOT_INCLUDED')


In [ ]:
# 3. Fiksni plan PRE racunanja buducih ishoda. Sacuvaj lokalno.
plan=make_micro_warning_plan(packet)
plan_path=private/'micro-warning-plan.json'
with plan_path.open('xb') as f:f.write(plan)
print('Plan SHA-256:',hashlib.sha256(plan).hexdigest())
files.download(str(plan_path))


In [ ]:
# 4. Deskriptivni Early Warning na svim dostupnim zatvorenim 5s bucketima.
report=run_micro_warning_study(packet,plan)
print('PROVJEREN MICRO REPORT; bez p-values, bez dokazanog edgea')
print('Procjenjivi feature intervali:',report['eligible_closed_5s_buckets'])
print('Ex-post reference episodes:',report['reference']['episodes'])
for t in report['trials']:
    print(t['rule'],f"{t['threshold_bps']}bps/{t['horizon_seconds']}s",
          f"alarms={t['total_alerts']}",f"correct={t['outcomes']['correct']}",
          f"false={t['false_alarms']}",f"unknown={t['outcomes']['unknown']}",t['status'])


In [ ]:
# 5. Ex-post pregled 5s intervala oko jedne referentne epizode.
# Izgled epizode NIJE prediktivan dokaz sam po sebi.
import matplotlib.pyplot as plt
if report['gallery']:
    case=report['gallery'][0];rows=case['timeline']
    xs=[(r['start_ms']-case['reference_ms'])/1000 for r in rows]
    fig,axes=plt.subplots(4,1,figsize=(13,10),sharex=True)
    series=[('last_trade_price','C2 zadnja trgovana cijena'),
            ('delta_base','C2 taker delta (BTC)'),
            ('quote_notional','C2 notional po 5s'),
            ('max_agg_fraction','Najveci aggTrade / notional')]
    for ax,(key,title) in zip(axes,series):
        ax.plot(xs,[r[key] for r in rows],marker='.',linewidth=1)
        ax.axvline(0,linestyle='--',label='ex-post referenca')
        ax.set_ylabel(title)
        ax.grid(True,alpha=.2)
    axes[-1].set_xlabel('Sekunde u odnosu na ex-post referentni prelaz (0)')
    fig.suptitle('Development mikrostruktura oko retrospektivnog pokreta; bez stvarnih L2 podataka')
    plt.tight_layout();plt.show()
else:
    print('Nema referentnih epizoda prema fiksnom testu; bez izmisljenog primjera.')


In [ ]:
# 6. Cjelokupan privatni izvjestaj, ukljucujuci negativne alarme.
result_path=private/'micro-warning-report.json'
with result_path.open('x',encoding='utf-8') as f:
    json.dump(report,f,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)
    f.write('\n')
files.download(str(result_path))
print('Sacuvaj plan + report privatno; nikada ne objavljuj u javnom repozitoriju.')
